# Customer Journey Markov Analysis 

In [ ]:

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display

EVENTS_PATH = r"C:\Users\samvi\Downloads\original_dataset\cleaned_dat_train1.csv"
EVENT_DEFS_PATH = None
USE_STAGE = False

JOURNEY_COL = "id"
EVENT_COL = "event_name"
TS_COL = "event_timestamp"
STEP_COL = "journey_steps_until_end"

SUCCESS_EVENT = "order_shipped"
SUCCESS_STATE = "__success_order_shipped__"
FAILED_STATE = "__failed__"
CENSOR_END_STATE = "__censored_end__"

INACTIVITY_DAYS = 60
INACTIVITY_STATE = f"__inactivity_gap_{INACTIVITY_DAYS}d__"

TOPK = 20
MIN_COUNT = 50
SNAPSHOT_TS_UTC = None

In [ ]:

def normalize_state(x):
    if pd.isna(x):
        return "unknown"
    s = str(x).strip().lower()
    s = re.sub(r"\s+", "_", s)
    return s


def choose_column(df, preferred, fallbacks):
    if preferred in df.columns:
        return preferred
    for c in fallbacks:
        if c in df.columns:
            return c
    raise KeyError(f"Could not find '{preferred}' or any of {fallbacks}. Columns={list(df.columns)}")


def read_event_defs(path):
    ed = pd.read_csv(path)
    ed.columns = [c.strip() for c in ed.columns]
    return ed


def build_event_to_stage_map(ed):
    cols = {c.lower(): c for c in ed.columns}
    event_col_candidates = ["event_name", "event", "name"]
    stage_col_candidates = ["stage", "stage_name", "funnel_stage", "category", "group"]

    event_col = next((cols[c] for c in event_col_candidates if c in cols), None)
    stage_col = next((cols[c] for c in stage_col_candidates if c in cols), None)

    if event_col is None or stage_col is None:
        return {}

    mapping = {}
    tmp = ed[[event_col, stage_col]].dropna()
    for _, row in tmp.iterrows():
        mapping[normalize_state(row[event_col])] = normalize_state(row[stage_col])
    return mapping


def to_transition_probs(counts):
    return counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)


def outgoing_entropy(P):
    ent = {}
    for s in P.index:
        p = P.loc[s].values
        p = p[p > 0]
        ent[s] = float(-(p * np.log(p)).sum()) if len(p) else 0.0
    return pd.Series(ent, name="outgoing_entropy")


def top_transitions(P, counts, k, min_count):
    rows = []
    for s in P.index:
        for t, p in P.loc[s].items():
            c = int(counts.loc[s, t]) if (s in counts.index and t in counts.columns) else 0
            if c >= min_count and p > 0:
                rows.append((s, t, float(p), c))
    out = pd.DataFrame(rows, columns=["from_state", "to_state", "prob", "count"])
    if out.empty:
        return out
    return out.sort_values(["prob", "count"], ascending=[False, False]).head(k)


def expected_steps_to_end(P, end_state):
    all_states = P.index.union(P.columns)
    P2 = P.reindex(index=all_states, columns=all_states, fill_value=0.0)

    if end_state not in P2.index:
        P2.loc[end_state] = 0.0
    if end_state not in P2.columns:
        P2[end_state] = 0.0

    P2.loc[end_state, :] = 0.0
    P2.loc[end_state, end_state] = 1.0

    row_sums = P2.sum(axis=1)
    nz = row_sums > 0
    P2.loc[nz] = P2.loc[nz].div(row_sums[nz], axis=0)

    transient = [s for s in P2.index if s != end_state]
    if len(transient) == 0:
        return pd.Series(dtype=float)

    Q = P2.loc[transient, transient].values
    I = np.eye(Q.shape[0])

    try:
        N = np.linalg.inv(I - Q)
    except np.linalg.LinAlgError:
        N = np.linalg.pinv(I - Q)

    t = (N @ np.ones((Q.shape[0], 1))).flatten()
    return pd.Series(t, index=transient, name=f"E_steps_until_{end_state}")

## Load and preprocess events

The PDF shows the notebook loading the event log CSV, normalizing event names, optionally mapping events to stages, sorting within journeys, and choosing the snapshot timestamp as the maximum observed timestamp when not manually specified. fileciteturn0file0L21-L57

In [ ]:

df = pd.read_csv(EVENTS_PATH)

JOURNEY_COL = choose_column(df, JOURNEY_COL, ["id"])
EVENT_COL = choose_column(df, EVENT_COL, ["event_name"])
TS_COL = choose_column(df, TS_COL, ["event_timestamp"])

df[TS_COL] = pd.to_datetime(df[TS_COL], errors="coerce", utc=True)
df["_event"] = df[EVENT_COL].map(normalize_state)

event_to_stage = {}
if EVENT_DEFS_PATH is not None:
    ed = read_event_defs(EVENT_DEFS_PATH)
    event_to_stage = build_event_to_stage_map(ed)
    if USE_STAGE and not event_to_stage:
        USE_STAGE = False

if USE_STAGE and event_to_stage:
    df["_state"] = df["_event"].map(lambda e: event_to_stage.get(e, "unknown_stage"))
    STATE_KIND = "stage"
else:
    df["_state"] = df["_event"]
    STATE_KIND = "event"

df = df.dropna(subset=[JOURNEY_COL, TS_COL, "_state"]).copy()

if STEP_COL in df.columns:
    df["_step"] = pd.to_numeric(df[STEP_COL], errors="coerce")
    df = df.sort_values([JOURNEY_COL, TS_COL, "_step"], kind="mergesort")
else:
    df = df.sort_values([JOURNEY_COL, TS_COL], kind="mergesort")

if SNAPSHOT_TS_UTC is None:
    snapshot_ts = df[TS_COL].max()
else:
    snapshot_ts = pd.to_datetime(SNAPSHOT_TS_UTC, utc=True)

In [ ]:

print("Loaded events:", len(df))
print("Unique journeys:", df[JOURNEY_COL].nunique())
print("Unique states:", df["_state"].nunique())
print("Snapshot time:", snapshot_ts)

In the PDF render, these summary values were:

- Loaded events: 51,848,861
- Unique journeys: 1,430,445
- Unique states: 27
- Snapshot time: `2023-01-23 12:29:56+00:00` fileciteturn0file0L58-L72

## Success truncation, inactivity gaps, and terminal-state construction

The next block in the PDF identifies the first success timestamp per journey, truncates journeys at first success, inserts an inactivity state when the inter-event gap exceeds 60 days, and assigns each journey to one of three terminal outcomes: success, failed, or censored. fileciteturn0file0L73-L111

In [ ]:

success_event_norm = normalize_state(SUCCESS_EVENT)
succ_mask = df["_event"] == success_event_norm

first_succ_ts = (
    df.loc[succ_mask]
    .groupby(JOURNEY_COL)[TS_COL]
    .min()
)

df = df.merge(first_succ_ts.rename("_first_success_ts"), on=JOURNEY_COL, how="left")
df = df[(df["_first_success_ts"].isna()) | (df[TS_COL] <= df["_first_success_ts"])]
df.loc[df["_event"] == success_event_norm, "_state"] = SUCCESS_STATE

gap_seconds = INACTIVITY_DAYS * 86400

df["_next_ts"] = df.groupby(JOURNEY_COL)[TS_COL].shift(-1)
df["_next_state_raw"] = df.groupby(JOURNEY_COL)["_state"].shift(-1)
df["_gap_sec"] = (df["_next_ts"] - df[TS_COL]).dt.total_seconds()

obs = df.dropna(subset=["_next_state_raw"])[[JOURNEY_COL, "_state", "_next_state_raw", "_gap_sec"]].copy()

rows = []
base_trans = obs.rename(columns={"_state": "from_state", "_next_state_raw": "to_state"})[
    ["from_state", "to_state"]
]
rows.append(base_trans)

gappy = obs[(obs["_gap_sec"].notna()) & (obs["_gap_sec"] > gap_seconds)].copy()
if len(gappy):
    rows.append(pd.DataFrame({
        "from_state": gappy["_state"].values,
        "to_state": INACTIVITY_STATE
    }))
    rows.append(pd.DataFrame({
        "from_state": INACTIVITY_STATE,
        "to_state": gappy["_next_state_raw"].values
    }))

transitions_df = pd.concat(rows, ignore_index=True)

last = df.groupby(JOURNEY_COL).tail(1)[[JOURNEY_COL, "_state", TS_COL]].copy()
last = last.rename(columns={"_state": "last_state", TS_COL: "last_ts"})

age_sec = (snapshot_ts - last["last_ts"]).dt.total_seconds()
is_success = last["last_state"] == SUCCESS_STATE
is_failed = (~is_success) & (age_sec > gap_seconds)
is_censored = (~is_success) & (~is_failed)

terminal_rows = []
if is_success.any():
    terminal_rows.append(pd.DataFrame({
        "from_state": [SUCCESS_STATE] * int(is_success.sum()),
        "to_state": [SUCCESS_STATE] * int(is_success.sum())
    }))
if is_failed.any():
    terminal_rows.append(pd.DataFrame({
        "from_state": last.loc[is_failed, "last_state"].values,
        "to_state": FAILED_STATE
    }))
if is_censored.any():
    terminal_rows.append(pd.DataFrame({
        "from_state": last.loc[is_censored, "last_state"].values,
        "to_state": CENSOR_END_STATE
    }))

if terminal_rows:
    terminal_df = pd.concat(terminal_rows, ignore_index=True)
else:
    terminal_df = pd.DataFrame(columns=["from_state", "to_state"])

transitions_all = pd.concat([transitions_df, terminal_df], ignore_index=True)

terminal_outcome = pd.Series(
    np.where(is_success, "success", np.where(is_failed, "failed", "censored")),
    index=last[JOURNEY_COL]
)

## Transition matrix and friction metrics

In [ ]:

counts = pd.crosstab(transitions_all["from_state"], transitions_all["to_state"]).astype(int)

for absorbing in [SUCCESS_STATE, FAILED_STATE, CENSOR_END_STATE]:
    if absorbing not in counts.index:
        counts.loc[absorbing] = 0
    if absorbing not in counts.columns:
        counts[absorbing] = 0
    counts.loc[absorbing, :] = 0
    counts.loc[absorbing, absorbing] = max(1, int(counts.loc[absorbing, absorbing]) if absorbing in counts.index else 1)

if INACTIVITY_STATE not in counts.index:
    counts.loc[INACTIVITY_STATE] = 0
if INACTIVITY_STATE not in counts.columns:
    counts[INACTIVITY_STATE] = 0

counts = counts.sort_index(axis=0).sort_index(axis=1)
P = to_transition_probs(counts)

print("States:", len(P))

topT = top_transitions(P, counts, k=TOPK, min_count=MIN_COUNT)
display(topT)

The PDF reports `States: 30` and shows the top observed transitions table, including strong self-loops such as `browse_products -> browse_products` and `application_web_view -> application_web_view`. fileciteturn0file0L112-L131

In [ ]:

rows = []
for s in P.index:
    p_self = float(P.loc[s, s]) if s in P.columns else 0.0
    self_count = int(counts.loc[s, s]) if (s in counts.index and s in counts.columns) else 0
    out_total = int(counts.loc[s].sum()) if s in counts.index else 0
    rows.append((s, p_self, 1.0 - p_self, self_count, out_total))

stuck = pd.DataFrame(
    rows,
    columns=["state", "p_self", "p_leave", "self_count", "outgoing_count"]
).set_index("state")

stuck = stuck.join(outgoing_entropy(P), how="left")

df["_dwell"] = (df["_next_ts"] - df[TS_COL]).dt.total_seconds()
dwell = df.dropna(subset=["_dwell"]).copy()
dwell = dwell[dwell["_dwell"] >= 0]

if len(dwell):
    g = dwell.groupby("_state")["_dwell"]
    dwell_stats = pd.DataFrame({
        "mean_dwell_seconds": g.mean(),
        "median_dwell_seconds": g.median(),
        "n_dwell_samples": g.size()
    })
    stuck = stuck.join(dwell_stats, how="left")

stuck_rank = stuck.drop(index=[SUCCESS_STATE, FAILED_STATE, CENSOR_END_STATE], errors="ignore").copy()
stuck_rank["friction_score"] = (
    2.0 * stuck_rank["p_self"]
    + 0.5 * np.log1p(stuck_rank["self_count"])
    + 0.5 * np.log1p(stuck_rank.get("mean_dwell_seconds", 0.0))
    + 0.5 * (1.0 / (1.0 + stuck_rank["outgoing_entropy"]))
)

display(stuck_rank.sort_values("friction_score", ascending=False).head(TOPK))

dead_end_last_state = last["last_state"].value_counts(normalize=True)
dead_end = dead_end_last_state.head(TOPK).to_frame("terminal_share")
display(dead_end)

terminal_share = terminal_outcome.value_counts(normalize=True).to_frame("share")
display(terminal_share)

According to the PDF tables, the largest terminal-outcome shares were approximately:

- failed: `0.694970`
- success: `0.194339`
- censored: `0.110691` fileciteturn0file0L132-L169

## Expected steps to endpoints

In [ ]:

E_censor = expected_steps_to_end(P, CENSOR_END_STATE)
E_success = expected_steps_to_end(P, SUCCESS_STATE)
E_fail = expected_steps_to_end(P, FAILED_STATE)

display(E_censor.sort_values(ascending=False).head(TOPK).to_frame())
display(E_success.sort_values(ascending=False).head(TOPK).to_frame())
display(E_fail.sort_values(ascending=False).head(TOPK).to_frame())

The PDF shows the longest expected step counts for `application_web_view`, `application_web_submit`, `browse_products`, `add_to_cart`, and related states when measuring distance to censor, success, and failure endpoints. It also displays some negative values for a few states in the censor/success tables, which likely reflect the exact linear-algebra setup used in the original notebook and the presence of special inserted states. fileciteturn0file0L170-L230

## One-step stopping probabilities

In [ ]:

def stop_probabilities_to(target_state):
    if target_state not in P.columns:
        return pd.Series(dtype=float)
    s = P[target_state].drop(index=[target_state], errors="ignore").sort_values(ascending=False)
    return s

p_stop_censor = stop_probabilities_to(CENSOR_END_STATE).head(TOPK)
p_stop_success = stop_probabilities_to(SUCCESS_STATE).head(TOPK)
p_stop_fail = stop_probabilities_to(FAILED_STATE).head(TOPK)

display(p_stop_censor.to_frame("p_stop_censor"))
display(p_stop_success.to_frame("p_stop_success"))
display(p_stop_fail.to_frame("p_stop_fail"))

The PDF's one-step stopping tables highlight:

- `site_registration` as the highest one-step censor-stop probability
- `account_downpaymentcleared` as by far the highest one-step success probability
- `application_phone_approved` as the highest one-step failure probability

These appear directly in the rendered tables and charts. fileciteturn0file0L231-L288

## Visualizations

In [ ]:

plt.figure()
topT_plot = topT.sort_values("count", ascending=False).head(TOPK)
plt.barh(topT_plot["from_state"] + "→" + topT_plot["to_state"], topT_plot["count"])
plt.gca().invert_yaxis()
plt.xlabel("Transition count")
plt.title("Top observed transitions by count")
plt.tight_layout()
plt.show()

plt.figure()
friction_plot = stuck_rank.sort_values("friction_score", ascending=False).head(TOPK)
plt.barh(friction_plot.index, friction_plot["friction_score"])
plt.gca().invert_yaxis()
plt.xlabel("Friction score")
plt.title("High-friction states")
plt.tight_layout()
plt.show()

plt.figure()
dead_plot = dead_end.copy().sort_values("terminal_share", ascending=True)
plt.barh(dead_plot.index, dead_plot["terminal_share"])
plt.xlabel("Share of journeys ending in state")
plt.title("Most common terminal states (last observed)")
plt.tight_layout()
plt.show()

plt.figure()
ts_plot = terminal_share.copy().sort_values("share", ascending=True)
plt.barh(ts_plot.index, ts_plot["share"])
plt.xlabel("Share of journeys")
plt.title("Outcome mix: success vs failed vs censored")
plt.tight_layout()
plt.show()

plt.figure()
ps_plot = p_stop_censor.sort_values(ascending=True)
plt.barh(ps_plot.index, ps_plot.values)
plt.xlabel("P(next = censored)")
plt.title("Highest one-step censor-stop probabilities")
plt.tight_layout()
plt.show()

plt.figure()
pss_plot = p_stop_success.sort_values(ascending=True)
plt.barh(pss_plot.index, pss_plot.values)
plt.xlabel("P(next = success)")
plt.title("Highest one-step success probabilities")
plt.tight_layout()
plt.show()

plt.figure()
psf_plot = p_stop_fail.sort_values(ascending=True)
plt.barh(psf_plot.index, psf_plot.values)
plt.xlabel("P(next = failed)")
plt.title("Highest one-step failure probabilities")
plt.tight_layout()
plt.show()

In [ ]:

threshold_count = MIN_COUNT
edges = []

for i, row in counts.iterrows():
    for j, c in row.items():
        if c >= threshold_count and i != j:
            p = float(P.loc[i, j]) if j in P.columns else 0.0
            if p > 0:
                edges.append((i, j, c, p))

edges_df = pd.DataFrame(edges, columns=["from_state", "to_state", "count", "prob"])

G = nx.DiGraph()
for _, r in edges_df.iterrows():
    G.add_edge(r["from_state"], r["to_state"], weight=r["prob"], count=r["count"])

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=0.6, iterations=50)
edge_widths = [G[u][v]["weight"] * 5 for u, v in G.edges()]

nx.draw_networkx_nodes(G, pos, node_size=600)
nx.draw_networkx_labels(G, pos, font_size=8)
nx.draw_networkx_edges(G, pos, width=edge_widths, arrows=True, arrowsize=10)

plt.title("Markov state transition graph (filtered by count)")
plt.axis("off")
plt.tight_layout()
plt.show()

## Notes

- This notebook is intended to reproduce the logic shown in the PDF, not necessarily the exact original execution environment.
- You will need the underlying CSV at `EVENTS_PATH` to re-run it.
- If you want, I can also produce a second version that is cleaned up for reuse on new datasets, with parameterized paths and some safeguards for large-memory workloads. The reconstruction here follows the PDF closely. fileciteturn0file0L1-L20